# Preparing inputs for a profiling simulation (pt. i: parent simulation)
Here we will prepare the inputs for a simulation to be used for scaling analysis and profiling, including all features supported by \[C\]Worthy's fork of UCLA-ROMS. In order to run the profiling simulation, we first need to run a pair of simulations in order to generate inputs: a parent, from which boundary conditions will be generated for a child, and then the child, which generates additional inputs for a second run of the parent.

Note that the simulation setup is effectively the same as the nested CDR tutorial in the ROMS documentation, but with the domain shifted southward such that there is no land.

In [ ]:
%matplotlib inline
import roms_tools as rt
from pathlib import Path
import datetime as dt

**After setting the next cell according to your own source data paths, the rest of the notebook should run:**

In [ ]:
# ROMS tools source data:
rtd=Path("~/Code/roms_tools_datasets")

topo_path = rtd/"SRTM15_V2.5.nc" # Topography (SRTM15)
era5_path = [rtd/"ERA5_2012-01.nc", rtd/"ERA5_2012-02.nc"] # Surface forcing (ERA5)
bgc_path = rtd/"BGC/BGCdataset.nc" # BGC tracers for initial and boundary conditions (CWorthy unified BGC dataset)
bgc_surf_path = rtd/"BGC/BGCdataset.nc" # BGC surface forcing (CWorthy unified BGC dataset)
glorys_paths = [rtd/f"GLORYS/mercatorglorys12v1_gl12_mean_2012{m:02}{d:02}.nc" for m in range(1,3) for d in range(1,32)][:-2] # Interior state for initial and boundary conditions (GLORYS)
tpxo_path = rtd/"TPXO10.v2/" # Tidal forcing (TPXO)

In [ ]:
# Set directory in which to save files
save_dir=f"{os.environ.get("ROMS_ROOT")}/profiling/input_data"

## Simulation timing

In [ ]:
forcing_start_time = dt.datetime(2012,1,1,0,0,0)
forcing_end_time = dt.datetime(2012,3,1,0,0,0)
ini_time = dt.datetime(2012,1,1,12,0,0)

## Simulation domains

### Preparing grid files and nesting files

In [ ]:
parent_grid = rt.Grid(
                        N=20,                       # Vertical levels
                        nx=192,                     # Points in x direction
                        ny=192,                     # Points in y direction
                        size_x=1750,                # Width of domain (km)
                        size_y=2500,                # Height of domain (km)
                        center_lon=-25,             # Center longitude
                        center_lat=51,              # Center latitude
                        rot=0,                      # Rotation
                        topography_source={         # Source topography data
                            "name": "SRTM15",
                            "path": str(topo_path),
                        }
                    )
parent_grid.plot()

In [ ]:
# Save grid data
parent_grid_path    = parent_grid.save(save_dir+"/parent_grid.nc")

## Simulation timing

In addition to the domain, the timing of the simulation is foundational to all of the other inputs. We need to provide `roms-tools` with start and end times that will cover all surface and boundary forcing files, and an initialization time, within that window, from which to begin the simulation. We must also set time-stepping parameters in ROMS.
We first set some local variables in this notebook to be used in `roms-tools`:

Here we have set the simulation to begin 12 hours after the beginning of the forcing time. _The simulation must always start at or after the beginning of the forcing._

## Surface forcing
We will generate two surface forcing datasets using `roms-tools` (for more information on the process, see the [corresponding documentation](https://roms-tools.readthedocs.io/en/latest/surface_forcing.html)). The first is a bulk forcing input governing the variables impacting the physical circulation (10m wind velocity, short-wave radiation, 2m air temperature, precipitation, 2m humidity). The second covers variables impacting the biogeochemical equations (pCO2, iron deposition, dust deposition, and optionally - and not used by us for this simulation - nitrogen fluxes). These two forcing datasets are controlled within ROMS by parameters set in the `bulk_frc.opt` and `bgc.opt` files, respectively. We'll first generate and save these files, then take a look at these option files.

### Bulk forcing

In [ ]:
# Physical surface forcing
parent_surface_bulk_forcing = rt.SurfaceForcing(
    grid = parent_grid,
    start_time= forcing_start_time,
    end_time= forcing_end_time,
    type= "physics",
    source={"name": "ERA5", "path": era5_path},
    use_dask=True,
)
# Save to netCDF
parent_surface_bulk_forcing_path = parent_surface_bulk_forcing.save(save_dir+"/parent_surface_bulk_forcing.nc", group=False)



### Biogeochemical surface forcing


In [ ]:
# BGC surface forcing
parent_bgc_surface_forcing = rt.SurfaceForcing(
    grid=parent_grid,
    start_time = forcing_start_time,
    end_time = forcing_end_time,
    source={"name": "UNIFIED", "path": bgc_surf_path, "climatology":True},
    type="bgc",
    use_dask=True,
)
parent_bgc_surface_forcing_path = parent_bgc_surface_forcing.save(save_dir+"/parent_bgc_surface_forcing.nc", group=False)


## Boundary forcing


In [ ]:
parent_phys_boundary_forcing = rt.BoundaryForcing(
    grid=parent_grid,
    start_time = forcing_start_time,
    end_time = forcing_end_time,
    source={"name": "GLORYS", "path": glorys_paths},
    type="physics",  # "physics" or "bgc"; default is "physics"
    use_dask=True,
)

parent_bgc_boundary_forcing = rt.BoundaryForcing(
    grid=parent_grid,
    start_time = forcing_start_time,
    end_time = forcing_end_time,
    source={"name": "UNIFIED", "path": bgc_path, "climatology": True},
    type="bgc",
    use_dask=True,
)

parent_phys_boundary_forcing_path = parent_phys_boundary_forcing.save(save_dir+"/parent_phys_boundary_forcing.nc",group=False)
parent_bgc_boundary_forcing_path  = parent_bgc_boundary_forcing.save(save_dir+"/parent_bgc_boundary_forcing.nc",group=False)

## Tidal Forcing


In [ ]:
tpxo_path = rtd/"TPXO10.v2/"

parent_tidal_forcing = rt.TidalForcing(
    grid=parent_grid,
    source={"name": "TPXO", "path": 
                {
                "grid": tpxo_path / "grid_tpxo10v2.nc",
                "h": tpxo_path / "h_tpxo10.v2.nc",
                "u": tpxo_path / "u_tpxo10.v2.nc",
                }
           },
    ntides=10,
    use_dask=True
)
parent_tidal_forcing.plot("ssh_Re", ntides=0)
parent_tidal_forcing_path = parent_tidal_forcing.save(save_dir+"/parent_tides.nc")


## Initial Conditions

In [ ]:
parent_initial_conditions = rt.InitialConditions(
    grid = parent_grid,
    ini_time = ini_time,
    source={"name": "GLORYS", "path": glorys_paths[0]},
    bgc_source = {
        "name": "UNIFIED",
        "path": bgc_path,
        "climatology": True,
        },
    use_dask = True,
)

parent_initial_conditions_path = parent_initial_conditions.save(save_dir+"/parent_initial_conditions.nc")


## Nesting-specific inputs and settings
As we intend for this simulation to generate boundary forcing and initial conditions for a nested subdomain, we must tell ROMS where that subdomain is, so we generate the grid for our child simulation now, and save information about the nesting relationship using `roms-tools`' `save_nesting` function:

In [ ]:
child_grid = rt.ChildGrid(
                        parent_grid=parent_grid, # Parent grid
                        N=20,                       # Vertical levels
                        nx=192,                     # Points in x direction
                        ny=192,                     # Points in y direction
                        size_x=400,                 # Width of domain (km)
                        size_y=400,                 # Height of domain (km)
                        center_lon=-25,             # Center longitude
                        center_lat=51,            # Center latitude
                        rot=10,                     # Rotation
                        topography_source={         # Source topography data
                            "name": "SRTM15",
                            "path": str(topo_path),
                        },
                        metadata={"include_bgc": True},
                    )
child_grid.plot_nesting(with_dim_names=True)

### Save nesting information and child grid file
We need the nesting information to run the parent and generate boundary conditions for the child:

In [ ]:
child_grid_nesting_path = child_grid.save_nesting(save_dir+"/child_grid_nesting_info.nc")

In [ ]:
child_grid.save(save_dir+"/child_grid.nc")